# Mini Project 1 — Analysis Notebook

**Your name:**  Riya Chaudhari

**Dataset:**  Open Brewery DB

**Date:**  May 19, 2026

This notebook has four sections. Work through them in order. Each section has instructions and a code cell to fill in. Add markdown cells to explain your thinking as you go — that writing is part of the assignment.

When you're done, publish this notebook to your GitHub repository and submit the URL to Canvas.

In [1]:
# Setup — run this cell first (required packages for MP1)
!pip install jupyter plotly kaleido pandas -q

import pandas as pd
import plotly.express as px

print("Setup complete.")


[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: python3.13 -m pip install --upgrade pip
error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try brew install
    xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a Python library that isn't in Homebrew,
    use a virtual environment:
    
    python3 -m venv path/to/venv
    source path/to/venv/bin/activate
    python3 -m pip install xyz
    
    If you wish to install a Python application that isn't in Homebrew,
    it may be easiest to use 'pipx install xyz', which will manage a
    virtual environment for you. You can install pipx with
    
    brew install pipx
    
    You may restore the old behavior of pip by passing
    the '--break-system-packages' flag to pip, or by adding
    'break-system-packages = true' to your pip.conf file. The latter
    will permanently disable this er

Setup complete.


---

## Section 1 — Overview

Before writing any code, fill in this section. A good Overview tells anyone reading your notebook — including a future employer — what the analysis is about before they see a single chart.

**Dataset:** [Open Brewery DB](https://www.openbrewerydb.org/) is a free, community-maintained catalog of breweries (API: `https://api.openbrewerydb.org/v1/breweries`). This MP1 folder includes a cleaned snapshot in `open_breweries_us.csv` (fields: `id`, `name`, `city`, `state`, `country`, `brewery_type`) so the notebook runs without network access. 

**Why this dataset:** I chose it to practice the full HCDE data pipeline — API acquisition, cleaning, pandas analysis, and visualization — on real geographic and categorical data that is easy to explore but still messy enough to require deliberate filtering and interpretation.

**Three analytical questions:**

1. What does the distribution of brewery types look like across different states?
2. Are certain types (micro, regional, brewpub) concentrated in particular regions?
3. Do cities with a higher number of breweries tend to have a more diverse mix of brewery types, or are they dominated by a single type?

**What a practitioner would do with these findings:** A regional planner, economic-development office, or brewery entrepreneur could use these patterns to see where certain brewery types cluster, which regions look saturated or underserved in this listing, and whether larger cities in the sample show more type diversity—useful context for siting, tourism messaging, or competitive research, with the caveat that conclusions apply to what Open Brewery DB lists, not every brewery in the country.

---

## Section 2 — Data Profile

Load your dataset and get a basic picture of what's in it. Answer these questions in a markdown cell below your code:

- How many rows and columns does your dataset have?
- What does each column represent?
- Are there any obvious data quality issues (missing values, unexpected types, inconsistent formatting)?
- Which column or columns will your analysis focus on, and why?

In [2]:
# Load from local CSV (standalone MP1 artifact; same cleaning as W5/W6)
from pathlib import Path

DATA_FILE = Path("open_breweries_us.csv")  # in this notebook folder (W7/)

raw_df = pd.read_csv(DATA_FILE)
cols = ["id", "name", "city", "state", "country", "brewery_type"]
raw_df = raw_df[[c for c in cols if c in raw_df.columns]]

df = raw_df[raw_df["country"] == "United States"].copy()
df = df.dropna(subset=["state", "city", "brewery_type"])
for col in ["state", "city", "brewery_type"]:
    df[col] = df[col].astype(str).str.strip()

print(f"Loaded {len(df):,} U.S. brewery rows from {DATA_FILE.name}")
print(df.shape)
df.head()

Loaded 4,310 U.S. brewery rows from open_breweries_us.csv
(4310, 6)


,id,name,city,state,country,brewery_type
0,5128df48-79fc-4f0f-8b52-d06be54d0cec,(405) Brewing Co,Norman,Oklahoma,United States,micro
1,9c5a66c8-cc13-416f-a5d9-0a769c87d318,(512) Brewing Co,Austin,Texas,United States,micro
2,34e8c68b-6146-453f-a4b9-1f6cd99a5ada,1 of Us Brewing Company,Mount Pleasant,Wisconsin,United States,micro
3,6d14b220-8926-4521-8d19-b98a2d6ec3db,10 Barrel Brewing Co,Bend,Oregon,United States,large
4,e2e78bd8-80ff-4a61-a65c-3bfbd9d76ce2,10 Barrel Brewing Co,Bend,Oregon,United States,large


In [3]:
# Check column types and missing values
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4310 entries, 0 to 4309
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   id            4310 non-null   str  
 1   name          4310 non-null   str  
 2   city          4310 non-null   str  
 3   state         4310 non-null   str  
 4   country       4310 non-null   str  
 5   brewery_type  4310 non-null   str  
dtypes: str(6)
memory usage: 202.2 KB


In [4]:
# Profile categorical fields (all columns are text identifiers / categories)
print("Missing values after cleaning:")
print(df.isna().sum())

print("\nBrewery type counts:")
print(df["brewery_type"].value_counts())

print(f"\nGeographic coverage: {df['state'].nunique()} states, {df['city'].nunique()} cities")

# For object columns, describe() reports count, unique, most common value, and frequency
df.describe(include="object")

Missing values after cleaning:
id              0
name            0
city            0
state           0
country         0
brewery_type    0
dtype: int64

Brewery type counts:
brewery_type
micro         2227
brewpub       1232
planning       322
closed         209
regional       102
contract        97
large           53
proprietor      31
taproom         22
nano            13
bar              1
location         1
Name: count, dtype: int64

Geographic coverage: 51 states, 1991 cities


/var/folders/tj/73s72wsd2xzfn8q6hlsrk2jc0000gn/T/ipykernel_69936/702501492.py:11: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df.describe(include="object")


,id,name,city,state,country,brewery_type
count,4310,4310,4310,4310,4310,4310
unique,4310,4221,1991,51,1,12
top,5128df48-79fc-4f0f-8b52-d06be54d0cec,Ballast Point Brewing Company,Portland,California,United States,micro
freq,1,6,52,494,4310,2227


**Your data profile notes:**

- **Rows and columns:** The cleaned snapshot `open_breweries_us.csv` in this folder has **4,310 rows and 6 columns** after filtering to U.S. breweries with non-missing `state`, `city`, and `brewery_type` (`id`, `name`, `city`, `state`, `country`, `brewery_type`). The source API snapshot contained roughly 5,000 rows before U.S. filtering; non-U.S. and incomplete rows were excluded when building the CSV.

- **What each column represents:** `id` is the API’s unique brewery identifier; `name` is the brewery name; `city` and `state` are location fields used for geographic grouping; `country` is the country label (all retained rows are `United States` after filtering); `brewery_type` is a categorical label such as `micro`, `brewpub`, or `regional` describing the kind of brewery.

- **Data quality issues:** In this pull, core fields were present for U.S. rows after the filter—no missing values in the cleaned frame. Important caveats still apply: the API is a **community-maintained listing**, not a complete census; counts can change between runs; some `brewery_type` values (for example `planning`, `closed`) are operational status rather than a style of beer business; and a few rare labels (`bar`, `location`) appear only once. Duplicate brewery names in the same city can occur (for example multiple “10 Barrel Brewing Co” rows in Bend, Oregon), which is worth remembering when interpreting city-level counts.

- **Columns for analysis:** The main focus is **`brewery_type`** (composition and concentration in Q1–Q2), together with **`state`** and **`city`** for geographic grouping. **`country`** is used only to restrict the sample to the U.S. `id` and `name` are kept for traceability but are not central to the three research questions. For Q2, `state` is mapped to U.S. Census regions in later code; for Q3, `city` and `state` together define each city bucket.

---

## Section 3 — Analysis

Answer your three research questions using pandas. Each question should have:

1. A markdown cell stating the question
2. A code cell with the analysis
3. A markdown cell with your interpretation — what does the result mean?

You may need to clean or reshape the data before you can answer a question. That's normal — document what you did and why.

**Question 1:** What does the distribution of brewery types look like across different states?

In [5]:
# Q1: count breweries by state and brewery_type
state_type = (
    df.groupby(["state", "brewery_type"])
    .size()
    .reset_index(name="count")
    .sort_values(["state", "count"], ascending=[True, False])
)

state_totals = df.groupby("state").size().sort_values(ascending=False)

print("Top 10 states by brewery count in this listing:")
print(state_totals.head(10).to_string())

print("\nType mix for the 8 largest states (state | brewery_type | count):")
top_states = state_totals.head(8).index
print(state_type[state_type["state"].isin(top_states)].to_string(index=False))


Top 10 states by brewery count in this listing:
state
California        494
Washington        268
Colorado          235
New York          220
Michigan          202
Texas             186
Florida           178
Pennsylvania      176
North Carolina    159
Oregon            144

Type mix for the 8 largest states (state | brewery_type | count):
       state brewery_type  count
  California        micro    262
  California      brewpub    126
  California     planning     35
  California       closed     19
  California     regional     19
  California     contract     14
  California        large     12
  California   proprietor      7
    Colorado        micro    120
    Colorado      brewpub     76
    Colorado     planning     15
    Colorado       closed      6
    Colorado     regional      6
    Colorado        large      5
    Colorado     contract      4
    Colorado   proprietor      3
     Florida        micro    111
     Florida     planning     28
     Florida      brewpub     26

**Interpretation:**  
`micro` and `brewpub` are the most common types in nearly every high-count state, which matches the overall dataset (micro is the single largest category nationwide). States with the most listed breweries—such as California, Colorado, and Washington—also show the largest raw counts for those types, so a tall total can reflect both “many breweries” and “many of the same common types.” The mix is not identical everywhere: some states show relatively more `planning` or `closed` rows, which are operational statuses rather than production styles. I would investigate next whether differences are about real market structure or about how completely each state is represented in Open Brewery DB.

**Question 2:** Are certain types (micro, regional, brewpub) concentrated in particular regions?

In [6]:
def map_us_region(state: str) -> str:
    """Map U.S. state names to Census-style regions (same logic as W5/W6)."""
    northeast = {
        "Connecticut", "Maine", "Massachusetts", "New Hampshire", "Rhode Island",
        "Vermont", "New Jersey", "New York", "Pennsylvania",
    }
    midwest = {
        "Illinois", "Indiana", "Michigan", "Ohio", "Wisconsin", "Iowa", "Kansas",
        "Minnesota", "Missouri", "Nebraska", "North Dakota", "South Dakota",
    }
    south = {
        "Delaware", "District of Columbia", "Florida", "Georgia", "Maryland",
        "North Carolina", "South Carolina", "Virginia", "West Virginia", "Alabama",
        "Kentucky", "Mississippi", "Tennessee", "Arkansas", "Louisiana", "Oklahoma", "Texas",
    }
    west = {
        "Arizona", "Colorado", "Idaho", "Montana", "Nevada", "New Mexico", "Utah",
        "Wyoming", "Alaska", "California", "Hawaii", "Oregon", "Washington",
    }
    if state in northeast:
        return "Northeast"
    if state in midwest:
        return "Midwest"
    if state in south:
        return "South"
    if state in west:
        return "West"
    return "Other/Unknown"


focus_types = ["micro", "regional", "brewpub"]
regional = df.copy()
regional["region"] = regional["state"].map(map_us_region)
regional = regional[regional["brewery_type"].isin(focus_types)]

region_concentration = (
    regional.groupby(["region", "brewery_type"])
    .size()
    .reset_index(name="count")
    .sort_values(["brewery_type", "count"], ascending=[True, False])
)
region_concentration["pct_within_type"] = region_concentration.groupby("brewery_type")["count"].transform(
    lambda s: (s / s.sum() * 100).round(2)
)

print("Counts and share of each type within region (pct is % of that type nationwide):")
print(region_concentration.to_string(index=False))


Counts and share of each type within region (pct is % of that type nationwide):
   region brewery_type  count  pct_within_type
     West      brewpub    391            31.74
  Midwest      brewpub    389            31.57
    South      brewpub    234            18.99
Northeast      brewpub    218            17.69
     West        micro    737            33.09
    South        micro    610            27.39
  Midwest        micro    479            21.51
Northeast        micro    401            18.01
     West     regional     40            39.22
    South     regional     23            22.55
  Midwest     regional     22            21.57
Northeast     regional     17            16.67


**Interpretation:**  
For all three focus types, the **West** and **Midwest** hold the largest raw counts, and the **Northeast** holds the smallest share of each type when measured as a percent within that type. That pattern partly reflects how many breweries are listed in each region overall, not necessarily a unique preference for one type in one region. `micro` and `brewpub` are far more numerous than `regional` everywhere, so “concentration” for regional breweries is harder to see at a glance without normalizing by total breweries per region. The takeaway: there are regional differences in where these types appear in the listing, but interpreting them as market preference requires comparing shares within each region, not raw bar heights alone.

**Question 3:** Do cities with a higher number of breweries tend to have a more diverse mix of brewery types, or are they dominated by a single type?

In [7]:
city_diversity = (
    df.groupby(["state", "city"])
    .agg(
        brewery_count=("id", "count"),
        unique_types=("brewery_type", "nunique"),
    )
    .reset_index()
)

city_diversity["dominant_type_share"] = (
    df.groupby(["state", "city", "brewery_type"])
    .size()
    .groupby(level=[0, 1])
    .apply(lambda s: round((s.max() / s.sum()) * 100, 2))
    .values
)

city_diversity = city_diversity.sort_values("brewery_count", ascending=False)
correlation = city_diversity["brewery_count"].corr(city_diversity["unique_types"])

print("Top 15 cities by brewery count (with type diversity):")
print(city_diversity.head(15).to_string(index=False))

print(
    f"\nPearson correlation between brewery_count and unique_types: {correlation:.3f}"
)
print("Higher unique_types and lower dominant_type_share suggest a more diverse mix.")


Top 15 cities by brewery count (with type diversity):
         state         city  brewery_count  unique_types  dominant_type_share
    California    San Diego             52             7                42.31
      Colorado       Denver             49             7                61.22
    Washington      Seattle             45             7                44.44
        Oregon     Portland             39             7                53.85
      Illinois      Chicago             33             6                45.45
         Texas       Austin             26             5                65.38
     Minnesota  Minneapolis             24             4                70.83
         Texas      Houston             22             4                63.64
       Indiana Indianapolis             20             3                50.00
      Michigan Grand Rapids             20             2                90.00
          Ohio     Columbus             20             6                50.00
     Wisco

**Interpretation:**  
Cities with more listed breweries generally also show **more distinct `brewery_type` values** (positive correlation between `brewery_count` and `unique_types`), so larger scenes in this sample are usually more diverse, not single-type only. That said, many large cities still have a **high `dominant_type_share`**—often 50–70% from one type (frequently `micro`)—so “more breweries” does not mean an even mix. Smaller cities with only one or two listings naturally sit at low counts and low diversity, so the most informative comparisons are among cities with many breweries (for example San Diego, Denver, Seattle). This supports association, not causation: the data show that scale and diversity move together in the listing, but not why.

---

## Section 4 — Visualization

Create at least one visualization that supports one of your analysis findings. Your chart should:

- Have a title that states the finding, not just the data (e.g., "Satisfaction scores drop sharply after age 40" not "Satisfaction by age")
- Have labeled axes
- Use a chart type appropriate for your data (bar for categorical comparison, scatter for relationships, line for trends over time)

Below the chart, explain in a markdown cell: why you chose this chart type, and what you want the reader to take away from it.

In [8]:
# Visualizations supporting Section 3 findings (uses df from Section 2)
# Reuses map_us_region from Question 2 if you already ran Section 3; otherwise define it here.
try:
    map_us_region
except NameError:
    def map_us_region(state: str) -> str:
        northeast = {"Connecticut", "Maine", "Massachusetts", "New Hampshire", "Rhode Island", "Vermont", "New Jersey", "New York", "Pennsylvania"}
        midwest = {"Illinois", "Indiana", "Michigan", "Ohio", "Wisconsin", "Iowa", "Kansas", "Minnesota", "Missouri", "Nebraska", "North Dakota", "South Dakota"}
        south = {"Delaware", "District of Columbia", "Florida", "Georgia", "Maryland", "North Carolina", "South Carolina", "Virginia", "West Virginia", "Alabama", "Kentucky", "Mississippi", "Tennessee", "Arkansas", "Louisiana", "Oklahoma", "Texas"}
        west = {"Arizona", "Colorado", "Idaho", "Montana", "Nevada", "New Mexico", "Utah", "Wyoming", "Alaska", "California", "Hawaii", "Oregon", "Washington"}
        if state in northeast: return "Northeast"
        if state in midwest: return "Midwest"
        if state in south: return "South"
        if state in west: return "West"
        return "Other/Unknown"

REGION_ORDER = ["Northeast", "Midwest", "South", "West", "Other/Unknown"]

# Chart 1 (Q1): stacked counts by state and brewery type
state_totals = df.groupby("state").size().sort_values(ascending=False)
top_states = state_totals.head(14).index.tolist()
sub = df[df["state"].isin(top_states)].copy()
sub["state"] = pd.Categorical(sub["state"], categories=top_states, ordered=True)

fig1 = px.histogram(
    sub,
    x="state",
    color="brewery_type",
    title="Micro and brewpub dominate the brewery mix in the busiest U.S. states",
    labels={"state": "State", "brewery_type": "Brewery type"},
    category_orders={"state": top_states},
)
fig1.update_layout(
    barmode="stack",
    xaxis_tickangle=-35,
    yaxis_title="Number of breweries",
    legend_title_text="Brewery type",
    height=520,
)
fig1.show()
fig1.write_image("chart_q1_brewery_types_by_state.png", scale=2)
print("Saved chart_q1_brewery_types_by_state.png")

# Chart 2 (Q2): grouped bars by Census region
focus_types = ["micro", "regional", "brewpub"]
regional = df[df["brewery_type"].isin(focus_types)].copy()
regional["region"] = regional["state"].map(map_us_region)
plot_df = (
    regional.groupby(["region", "brewery_type"], observed=True)
    .size()
    .reset_index(name="brewery_count")
)
plot_df["region"] = pd.Categorical(plot_df["region"], categories=REGION_ORDER, ordered=True)

fig2 = px.bar(
    plot_df,
    x="region",
    y="brewery_count",
    color="brewery_type",
    barmode="group",
    title="Micro and brewpub listings are concentrated in the West and Midwest, not the Northeast",
    labels={"region": "U.S. Census region", "brewery_count": "Number of breweries", "brewery_type": "Brewery type"},
    category_orders={"region": REGION_ORDER},
)
fig2.update_layout(legend_title_text="Brewery type", height=480)
fig2.show()
fig2.write_image("chart_q2_micro_regional_brewpub_by_region.png", scale=2)
print("Saved chart_q2_micro_regional_brewpub_by_region.png")

# Chart 3 (Q3): scatter — city scale vs type diversity
type_counts = df.groupby(["state", "city", "brewery_type"], observed=True).size().reset_index(name="n_per_type")
city_totals = type_counts.groupby(["state", "city"], observed=True)["n_per_type"].transform("sum")
type_counts["pct_of_city"] = (type_counts["n_per_type"] / city_totals * 100).round(2)
dominant = type_counts.groupby(["state", "city"], observed=True)["pct_of_city"].max().reset_index(name="dominant_type_share_pct")
city_summary = (
    df.groupby(["state", "city"], observed=True)
    .agg(brewery_count=("id", "count"), unique_types=("brewery_type", "nunique"))
    .reset_index()
    .merge(dominant, on=["state", "city"], how="left")
)

fig3 = px.scatter(
    city_summary,
    x="brewery_count",
    y="unique_types",
    color="dominant_type_share_pct",
    title="Larger cities list more brewery types, but one type still often dominates",
    labels={
        "brewery_count": "Breweries in city",
        "unique_types": "Distinct brewery types",
        "dominant_type_share_pct": "Share of largest type (%)",
    },
    color_continuous_scale="RdYlGn_r",
    opacity=0.65,
)
fig3.update_layout(coloraxis_colorbar=dict(title="Largest type<br>% of city"), height=520)
fig3.show()
fig3.write_image("chart_q3_city_brewery_count_vs_type_diversity.png", scale=2)
print("Saved chart_q3_city_brewery_count_vs_type_diversity.png")


Saved chart_q1_brewery_types_by_state.png


Saved chart_q2_micro_regional_brewpub_by_region.png


Saved chart_q3_city_brewery_count_vs_type_diversity.png


**Chart rationale:**

- **Chart 1 (stacked histogram):** Each bar is a state; bar height is total listed breweries; color shows how that total splits across `brewery_type`. I chose a stacked layout because Q1 is about **composition within states**, not a single winner-take-all category. **Takeaway:** In high-count states, `micro` and `brewpub` make up most of the stack; the chart makes that mix visible at a glance (limited to the 14 busiest states so labels stay readable).

- **Chart 2 (grouped bar chart):** After mapping states to Census regions, I plot raw counts for `micro`, `regional`, and `brewpub` side by side within each region. Grouped bars support **within-region comparison** (which type is largest in the Midwest vs the West). **Takeaway:** All three types appear most often in the West and Midwest; the Northeast has the smallest counts for each—readers should compare bar heights **within** a region, not only across regions, because total listing volume differs.

- **Chart 3 (scatter plot):** Each point is a city; x-axis is brewery count, y-axis is distinct types, and color is the share of the largest type in that city. A scatter fits Q3 because it shows **association between two numeric measures** plus dominance as a third cue. **Takeaway:** Cities with more listings usually have more distinct types (upward trend), but warm colors show many large cities are still dominated by one type—scale and diversity move together, but not as a perfectly even mix.

---

## Section 5 — Conclusions

Write 3–5 sentences summarizing what you found. Address these questions:

- What is the most important thing your analysis revealed?
- What surprised you?
- What would you investigate next if you had more time or data?
- What are the limitations of this analysis — what can't you conclude from this data?

Then complete the competency claim below.

**Summary of findings:**

The most important pattern in this Open Brewery DB sample is that **`micro` and `brewpub` listings dominate** geographically—across busy states and across Census regions—while `regional` breweries are much rarer. At the city level, **more listed breweries usually means more distinct types** (correlation ≈ 0.78), which surprised me less than the fact that many large cities still show **50–70% of listings from one type**, so “bigger scene” does not mean an even mix. If I had more time, I would normalize counts by population or total breweries per region, separate operational statuses (`planning`, `closed`) from production types, and compare this API snapshot to an independent source. The main limitation is that this is a **community-maintained listing**, not a complete census: I can describe patterns in what is listed, but I cannot claim full market coverage, causation, or trends over time without additional data.

---

## Competency Claim

Full competency claims for **C3, C5, C6, and C7** are in [`mp1.md`](mp1.md) in this folder, with evidence tied to this notebook and earlier weeks (W4–W6).